In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: required_first_run
# Competition-safe: No — learning profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
import torch
torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
DEVICE = torch.device("cuda" if RUNTIME_PROFILE == "gpu" else "cpu")
if RUNTIME_PROFILE == "gpu" and not torch.cuda.is_available(): raise RuntimeError("GPU profile requested but CUDA is unavailable")
torch.set_default_device(DEVICE)
_device_probe = (torch.ones(8, device=DEVICE) @ torch.ones(8, device=DEVICE)).item()
print(f"Compute device: {DEVICE}; probe={_device_probe:.1f}")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Pre-trained Encoders

Fast/offline path kiểm tra tokenizer–embedding shapes bằng vocabulary nhỏ; full online path chỉ tải khi người học chủ động.

In [ ]:
offline=FAST_MODE or os.getenv("OAI_OFFLINE","0")=="1"
text="artificial intelligence is useful"
if offline:
    vocab={token:i+1 for i,token in enumerate(sorted(set(text.split())))}; ids=torch.tensor([[vocab[t] for t in text.split()]])
    encoder=torch.nn.Embedding(len(vocab)+1,8); hidden=encoder(ids); cls=hidden[:,0]
    print("toy offline ids/hidden/cls",ids.shape,hidden.shape,cls.shape)
else:
    from transformers import AutoTokenizer,AutoModel
    model_name="bert-base-uncased"
    try:
        tokenizer=AutoTokenizer.from_pretrained(model_name); model=AutoModel.from_pretrained(model_name)
    except OSError as exc:
        raise RuntimeError("Model cache is missing. Connect once or set OAI_FAST_MODE=1 for the offline smoke path.") from exc
    inputs=tokenizer(text,return_tensors="pt")
    with torch.no_grad(): hidden=model(**inputs).last_hidden_state
    cls=hidden[:,0]; print(inputs["input_ids"].shape,hidden.shape,cls.shape)
assert cls.ndim==2